# 阶段 1 · 数据探查与质量校验

把数据说清楚：1 亿行、9 天窗口、字段质量、以及**采样偏差**这个最大的坑。

> 代码逻辑统一放在 `src/`，notebook 只做编排与解读。

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import config as C
import db
C.setup_chinese_font()
%matplotlib inline

## 1.1 读取方式基准测试

对比 pandas chunksize / pyarrow 流式 / DuckDB 全量扫描，选定主链路。

In [ ]:
import stage1_overview as s1
bench = s1.bench_read_methods()
bench

## 1.2 全量基础指标（DuckDB 直连 3.6 GB CSV）

In [ ]:
con = db.connect()
metrics = s1.basic_metrics(con)
metrics

## 1.2' 逐日行数 / 行为构成

In [ ]:
daily = s1.rows_per_day(con)
behav = s1.behavior_profile(con)
behav

## 1.3 质量检查：缺失、重复、越界、取值集合、异常高频用户

In [ ]:
quality = s1.quality_checks(con)
quality

## 1.4 分布检查：用户活跃度长尾与购买集中度

In [ ]:
dist = s1.activity_distribution(con)
dist['buckets']

## 1.5 采样偏差实证：前 100 万行 vs 分层采样 1%

**这是本项目最关键的一次自检。**

In [ ]:
cmp_df, _ = s1.sampling_bias(con)
cmp_df

## 1.6 产出报告

In [ ]:
s1.build_report(bench, metrics, daily, behav, quality, dist, cmp_df)
con.close()
# -> reports/data_quality_report.md